# Ders 1B — İlk Random Forest Baseline

Her iki veri seti için ayrı Random Forest modeli eğitilir. Bu model sonraki geliştirme adımları için baseline/gatekeeper olarak kullanılır.

## 1. Paket kurulumu

Bu hücre gerekli paketleri kontrol eder ve eksik olanları kurar. RDKit, SMILES metninden moleküler fingerprint ve descriptor üretmek için kullanılır.

In [ ]:
import sys, subprocess, importlib.util

def install_if_missing(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f'[INSTALL] {pip_name}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

for import_name, pip_name in [('pandas','pandas'),('numpy','numpy'),('matplotlib','matplotlib'),('sklearn','scikit-learn'),('joblib','joblib')]:
    install_if_missing(import_name, pip_name)

if importlib.util.find_spec('rdkit') is None:
    try: install_if_missing('rdkit','rdkit')
    except Exception: install_if_missing('rdkit','rdkit-pypi')
print('Paket kontrolü tamamlandı.')

## 2. Paketleri çağırma

Bu hücre temel veri işleme, grafik ve RDKit paketlerini çağırır.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from rdkit import Chem, DataStructs
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator, Descriptors
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator
print('Temel importlar tamamlandı.')

## 3. Genel ayarlar

Bu hücre veri linklerini, kolon adlarını, modelleme sabitlerini ve çıktı klasörünü tanımlar. İki veri seti aynı akış içinde ayrı ayrı işlenir.

In [ ]:
DATASETS = {
    'ERa_BLA_assay': {
        'url': 'https://github.com/MOL-FEST/MOL_FEST_2026/blob/main/Train_2_1_A14_A17_ERa_BLA_agonist_antagonist.csv',
        'model_prefix': 'model_ERa_BLA',
        'short_name': 'ERα BLA'
    },
    'ERa_LUC_VM7_assay': {
        'url': 'https://github.com/MOL-FEST/MOL_FEST_2026/blob/main/Train_2_2_A15_A18_ERa_LUC_VM7_agonist_antagonist.csv',
        'model_prefix': 'model_ERa_LUC_VM7',
        'short_name': 'ERα LUC VM7'
    },
}
TARGET_COLUMN = 'binary_label_agonist1_antagonist0'
SMILES_COLUMN = 'QSAR-Ready SMILES'
RANDOM_STATE = 42
TEST_SIZE = 0.20
MORGAN_BITS = 1024
MORGAN_RADIUS = 2
N_ESTIMATORS_FAST = 200
HGB_MAX_ITER_FAST = 80
OUTPUT_ROOT = Path('molfest_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Genel ayarlar hazır.')
print('Çıktı klasörü:', OUTPUT_ROOT.resolve())

## 4. Veri okuma fonksiyonları

Bu fonksiyonlar GitHub linklerini raw CSV formatına çevirir, CSV dosyalarını okur, target/SMILES kolonlarını bulur ve eksik satırları temizler.

In [ ]:
def ensure_output_root():
    global OUTPUT_ROOT
    if 'OUTPUT_ROOT' not in globals(): OUTPUT_ROOT = Path('molfest_outputs')
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    return OUTPUT_ROOT

def note(title, message=''):
    print('\n' + '='*90); print(title); print('='*90)
    if message: print(message)

def github_to_raw(url):
    return url.replace('https://github.com/','https://raw.githubusercontent.com/').replace('/blob/','/')

def read_csv_flexible(url):
    source = github_to_raw(url)
    for sep in [';', ',']:
        df = pd.read_csv(source, sep=sep, encoding='utf-8-sig', low_memory=False)
        if df.shape[1] > 1: return df, sep, source
    raise ValueError('CSV okunamadı. Link veya ayraç kontrol edilmeli.')

def detect_column(df, preferred, keywords):
    if preferred in df.columns: return preferred
    for col in df.columns:
        if any(k.lower() in col.lower() for k in keywords): return col
    raise ValueError(f'Kolon bulunamadı: {preferred}')

def load_one_dataset(dataset_key):
    info = DATASETS[dataset_key]
    df, sep, source = read_csv_flexible(info['url'])
    target_col = detect_column(df, TARGET_COLUMN, ['binary_label','label','target','class'])
    smiles_col = detect_column(df, SMILES_COLUMN, ['smiles'])
    note(f"{info['short_name']} verisi okundu", f"Satır sayısı: {df.shape[0]}\nKolon sayısı: {df.shape[1]}\nAyraç: {repr(sep)}\nTarget kolonu: {target_col}\nSMILES kolonu: {smiles_col}")
    return df, target_col, smiles_col, info

def clean_target_and_smiles(df, target_col, smiles_col):
    y_numeric = pd.to_numeric(df[target_col], errors='coerce')
    mask = y_numeric.notna() & df[smiles_col].notna()
    df_clean = df.loc[mask].copy().reset_index(drop=True)
    y = pd.to_numeric(df_clean[target_col], errors='coerce').astype(int).to_numpy()
    classes = np.unique(y)
    if len(classes) != 2: raise ValueError(f'Binary target bekleniyor. Bulunan sınıflar: {classes}')
    if not set(classes).issubset({0,1}):
        mapping={classes[0]:0, classes[1]:1}; y=np.array([mapping[v] for v in y], dtype=int)
    note('Target ve SMILES temizlendi', f"Temiz satır sayısı: {len(df_clean)}\nÇıkarılan satır sayısı: {len(df)-len(df_clean)}\nSınıf dağılımı: {dict(pd.Series(y).value_counts().sort_index())}")
    return df_clean, y

def load_all_datasets():
    loaded={}
    for key in DATASETS:
        df,t,s,info=load_one_dataset(key)
        df_clean,y=clean_target_and_smiles(df,t,s)
        loaded[key]={'df':df_clean,'y':y,'target_col':t,'smiles_col':s,'info':info}
    return loaded
print('Veri okuma fonksiyonları hazır.')

## 5. Feature üretim fonksiyonları

SMILES metni makine öğrenmesi modeli için doğrudan kullanılmaz. Bu fonksiyonlar SMILES verisini Morgan, MACCS ve RDKit descriptor temsillerine çevirir.

In [ ]:
def _valid_molecules(smiles):
    mols=[]; valid=[]
    for smi in smiles:
        mol=Chem.MolFromSmiles(str(smi)); mols.append(mol); valid.append(mol is not None)
    return mols, np.array(valid, dtype=bool)

def smiles_to_morgan(smiles):
    mols, valid = _valid_molecules(smiles)
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=MORGAN_BITS)
    names=[f'Morgan_{i}' for i in range(MORGAN_BITS)]; rows=[]
    for mol, keep in zip(mols, valid):
        if keep:
            fp=generator.GetFingerprint(mol); arr=np.zeros((MORGAN_BITS,), dtype=np.float32); DataStructs.ConvertToNumpyArray(fp, arr); rows.append(arr)
    return np.vstack(rows), names, valid

def smiles_to_maccs(smiles):
    mols, valid = _valid_molecules(smiles)
    names=[f'MACCS_{i}' for i in range(1,167)]; rows=[]
    for mol, keep in zip(mols, valid):
        if keep:
            fp=MACCSkeys.GenMACCSKeys(mol); arr=np.zeros((167,), dtype=np.float32); DataStructs.ConvertToNumpyArray(fp, arr); rows.append(arr[1:])
    return np.vstack(rows), names, valid

def smiles_to_rdkit_descriptors(smiles):
    mols, valid = _valid_molecules(smiles)
    names=[name for name,_ in Descriptors._descList]
    calc=MolecularDescriptorCalculator(names); rows=[]
    for mol, keep in zip(mols, valid):
        if keep:
            desc=np.array(calc.CalcDescriptors(mol), dtype=np.float32); rows.append(np.nan_to_num(desc, nan=0.0, posinf=0.0, neginf=0.0))
    return np.vstack(rows), names, valid

def build_features(df, y, smiles_col, feature_set='morgan'):
    smiles=df[smiles_col].tolist()
    if feature_set=='morgan': X,names,valid=smiles_to_morgan(smiles)
    elif feature_set=='maccs': X,names,valid=smiles_to_maccs(smiles)
    elif feature_set=='rdkit': X,names,valid=smiles_to_rdkit_descriptors(smiles)
    elif feature_set=='maccs_morgan':
        X1,n1,v1=smiles_to_maccs(smiles); X2,n2,v2=smiles_to_morgan(smiles)
        if not np.array_equal(v1,v2): raise ValueError('MACCS ve Morgan maskeleri farklı çıktı.')
        X,names,valid=np.hstack([X1,X2]), n1+n2, v1
    elif feature_set=='maccs_rdkit':
        X1,n1,v1=smiles_to_maccs(smiles); X2,n2,v2=smiles_to_rdkit_descriptors(smiles)
        if not np.array_equal(v1,v2): raise ValueError('MACCS ve RDKit maskeleri farklı çıktı.')
        X,names,valid=np.hstack([X1,X2]), n1+n2, v1
    elif feature_set=='morgan_rdkit':
        X1,n1,v1=smiles_to_morgan(smiles); X2,n2,v2=smiles_to_rdkit_descriptors(smiles)
        if not np.array_equal(v1,v2): raise ValueError('Morgan ve RDKit maskeleri farklı çıktı.')
        X,names,valid=np.hstack([X1,X2]), n1+n2, v1
    else: raise ValueError('Geçersiz feature_set')
    df_valid=df.loc[valid].reset_index(drop=True); y_valid=y[valid]
    note(f'Feature üretildi: {feature_set}', f"Molekül sayısı: {X.shape[0]}\nFeature sayısı: {X.shape[1]}\nİlk 20 feature: {names[:20]}")
    return X, y_valid, names, df_valid

def build_features_for_all(loaded, feature_set='morgan'):
    result={}
    for key,data in loaded.items():
        X,y,names,df_valid=build_features(data['df'], data['y'], data['smiles_col'], feature_set)
        result[key]={**data,'X':X,'y':y,'feature_names':names,'df':df_valid}
    return result
print('Feature üretim fonksiyonları hazır.')

## 6. Modelleme paketleri

Bu hücre model eğitimi, veri bölme, metrik hesaplama ve grafik çizimi için kullanılan scikit-learn bileşenlerini çağırır.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
print('Modelleme importları tamamlandı.')

## Metriklerin anlamı

- **TP / True Positive:** Gerçek sınıf 1, model tahmini 1.
- **TN / True Negative:** Gerçek sınıf 0, model tahmini 0.
- **FP / False Positive:** Gerçek sınıf 0, model tahmini 1.
- **FN / False Negative:** Gerçek sınıf 1, model tahmini 0.

Formüller:

```text
Recall = TP / (TP + FN)
Specificity = TN / (TN + FP)
Precision = TP / (TP + FP)
F1 = 2 × Precision × Recall / (Precision + Recall)
Balanced Accuracy = (Recall + Specificity) / 2
```

**ROC-AUC**, sınıfları ayırma gücünü farklı eşik değerleri üzerinden özetler. **AP**, pozitif sınıfı yakalama kalitesini precision-recall mantığıyla özetler.

## 7. Metrik ve model fonksiyonları

Bu bölüm metrik hesaplama, tahmin kaydetme ve ana aday modelleri üretmek için kullanılır. Ana aday modeller: RandomForest, ExtraTrees ve HistGradientBoosting.

In [ ]:
def get_score_class1(model, X):
    if hasattr(model, 'predict_proba'): return model.predict_proba(X)[:,1]
    if hasattr(model, 'decision_function'): return model.decision_function(X)
    return model.predict(X).astype(float)

def calculate_metrics(y_true, y_pred, y_score):
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    specificity = tn/(tn+fp) if (tn+fp) else np.nan
    return {'ROC':roc_auc_score(y_true,y_score), 'AP':average_precision_score(y_true,y_score), 'F1':f1_score(y_true,y_pred,zero_division=0), 'Accuracy':accuracy_score(y_true,y_pred), 'BalancedAccuracy':balanced_accuracy_score(y_true,y_pred), 'Recall':recall_score(y_true,y_pred,zero_division=0), 'Specificity':specificity, 'Precision':precision_score(y_true,y_pred,zero_division=0), 'TN':int(tn), 'FP':int(fp), 'FN':int(fn), 'TP':int(tp)}

def save_prediction_table(outdir, name, df_test, smiles_col, y_test, y_pred, y_score):
    outdir=Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    pred=pd.DataFrame({'SMILES':df_test[smiles_col].values, 'y_true':y_test, 'y_pred':y_pred, 'y_score_class1':y_score})
    path=outdir/f'{name}_predictions.csv'; pred.to_csv(path,index=False); print(f'[Kaydedildi] {path}'); return pred
print('Metrik ve kayıt fonksiyonları hazır.')

In [ ]:
def make_model(model_type, random_state=RANDOM_STATE):
    if model_type=='RandomForest':
        return RandomForestClassifier(n_estimators=N_ESTIMATORS_FAST, max_features='sqrt', class_weight='balanced_subsample', random_state=random_state, n_jobs=-1)
    if model_type=='ExtraTrees':
        return ExtraTreesClassifier(n_estimators=N_ESTIMATORS_FAST, max_features='sqrt', class_weight='balanced', random_state=random_state, n_jobs=-1)
    if model_type=='HistGradientBoosting':
        return HistGradientBoostingClassifier(max_iter=HGB_MAX_ITER_FAST, learning_rate=0.08, random_state=random_state)
    raise ValueError(f'Bilinmeyen model_type: {model_type}')

def candidate_model_types(): return ['RandomForest','ExtraTrees','HistGradientBoosting']

def fit_evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train,y_train); y_pred=model.predict(X_test); y_score=get_score_class1(model,X_test); return model,y_pred,y_score,calculate_metrics(y_test,y_pred,y_score)
print('Model üretim fonksiyonları hazır.')

## 8. Random Forest baseline

Random Forest baseline/gatekeeper model olarak kullanılır. Sonraki geliştirme adımları bu baseline ile karşılaştırılır.

In [ ]:
lesson_out=ensure_output_root()/'02_first_random_forest'; models_dir=lesson_out/'saved_models'; models_dir.mkdir(parents=True, exist_ok=True)
loaded=load_all_datasets(); features_all=build_features_for_all(loaded, feature_set='morgan'); rows=[]
for dataset_key,data in features_all.items():
    info=data['info']; model_name=f"{info['model_prefix']}_rf_baseline"; note(f'RF baseline eğitiliyor: {model_name}')
    X_train,X_test,y_train,y_test,df_train,df_test=train_test_split(data['X'],data['y'],data['df'],test_size=TEST_SIZE,stratify=data['y'],random_state=RANDOM_STATE)
    model,y_pred,y_score,met=fit_evaluate_model(make_model('RandomForest'),X_train,X_test,y_train,y_test)
    met.update({'Dataset':dataset_key,'ModelName':model_name,'ModelType':'RandomForest'}); rows.append(met)
    joblib.dump({'model':model,'feature_names':data['feature_names']}, models_dir/f'{model_name}.joblib')
    save_prediction_table(lesson_out,model_name,df_test,data['smiles_col'],y_test,y_pred,y_score)
    ConfusionMatrixDisplay.from_predictions(y_test,y_pred,display_labels=['class 0','class 1']); plt.title(f'{model_name} — Confusion Matrix'); plt.tight_layout(); plt.savefig(lesson_out/f'{model_name}_confusion_matrix.png',dpi=300,bbox_inches='tight'); plt.show()
    RocCurveDisplay.from_predictions(y_test,y_score); plt.title(f'{model_name} — ROC Curve'); plt.tight_layout(); plt.savefig(lesson_out/f'{model_name}_roc_curve.png',dpi=300,bbox_inches='tight'); plt.show()
    print(pd.Series(met).to_string())
metrics_df=pd.DataFrame(rows).sort_values('ROC',ascending=False); metrics_df.to_csv(lesson_out/'rf_baseline_metrics_both_datasets.csv',index=False); display(metrics_df)